# Self-attention from scratch

This notebook builds self-attention one idea at a time, in the same order as Lecture 8 of [CS231n](https://cs231n.stanford.edu/slides/2026/lecture_8.pdf) and with the same notation. Each step adds a single change to the previous one, and the slide numbers are given so that you can follow along.

| Step | Idea | Slides |
|---|---|---|
| 1 | Attention in a seq2seq RNN | 13-29 |
| 2 | The attention layer, with one query | 30-35 |
| 3 | Scaled dot product | 36-38 |
| 4 | Multiple queries | 39 |
| 5 | Keys and values | 40-45 |
| 6 | Cross-attention | 46 |
| 7 | Self-attention | 47-51 |
| 8 | Permutation equivariance | 52-58 |
| 9 | Positional encoding | 59-60 |
| 10 | Masked self-attention | 61-62 |
| 11 | Multi-headed self-attention | 63-67 |
| 12 | Four matrix multiplies | 68-77 |
| 13 | Memory cost | 78-79 |
| 14 | A PyTorch module | |

All the weights here are random: the goal is to see what each operation computes. The notebook `Transformer from Scratch` uses these pieces to train a model.

**Aprendizado Profundo para Reconhecimento Visual** — Ifes Campus Serra
Autor: Thiago M. Paixão

In [ ]:
# If True, test is running on Colab. Otherwise, test is assumed to be offline.
TEST_ON_COLAB = True
FOLDERNAME = 'Colab Notebooks/Aprendizado Profundo para Reconhecimento Visual/08_transformers'  # only used if TEST_ON_COLAB is True

assert not (FOLDERNAME is None and TEST_ON_COLAB), "FOLDERNAME has to be set if TEST_ON_COLAB is True"

In [ ]:
# Create dataset directory
import os

if TEST_ON_COLAB:
    # This mounts your Google Drive to the Colab VM.
    from google.colab import drive

    drive.mount('/content/drive')
    os.chdir(f'/content/drive/My Drive/{FOLDERNAME}')

In [ ]:
import math

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
torch.set_printoptions(precision=3, sci_mode=False)


def show_matrix(matrix, rows, columns, title, xlabel="", ylabel=""):
    fig, ax = plt.subplots(figsize=(4, 3.5))
    ax.imshow(matrix.detach(), cmap="viridis", vmin=0, vmax=1)
    ax.set_xticks(range(len(columns)), columns)
    ax.set_yticks(range(len(rows)), rows)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    plt.show()

## Step 1: attention in a seq2seq RNN (slides 13-29)

The lecture starts where the RNN lecture stopped. An encoder reads `we see the sky` and produces one hidden state $h_i$ for each word. With a single context vector, the decoder would have to squeeze the whole sentence into one vector. Attention replaces it with a different context vector at each decoder step.

Starting from the decoder state $s_{t-1}$:

1. Compute a scalar **alignment score** for each input: $e_{t,i} = f_{att}(s_{t-1}, h_i)$, where $f_{att}$ is a linear layer.
2. Normalize the scores to get **attention weights**: $a_t = \text{softmax}(e_t)$, so that $0 < a_{t,i} < 1$ and $\sum_i a_{t,i} = 1$.
3. Compute the **context vector** as a weighted sum: $c_t = \sum_i a_{t,i} h_i$.

We use random vectors in place of the RNN states.

In [ ]:
words = ["we", "see", "the", "sky"]
D = 8

h = torch.randn(len(words), D)  # encoder hidden states h1..h4
s0 = torch.randn(D)  # initial decoder state

# f_att is a linear layer that receives the pair (s, h_i)
f_att = nn.Linear(2 * D, 1)

# 1. Alignment scores: one scalar per input
e = torch.stack([f_att(torch.cat([s0, h_i])).squeeze() for h_i in h])

# 2. Attention weights
a = torch.softmax(e, dim=0)

# 3. Context vector
c = (a[:, None] * h).sum(dim=0)

print(f"scores e:  {e.detach()}")
print(f"weights a: {a.detach()}, sum = {a.sum():.3f}")
print(f"context c has shape {tuple(c.shape)}")

**A general operator is hiding here (slides 28-29).** Forget the RNN for a moment. There are *query* vectors (the decoder states) and *data* vectors (the encoder states). Each query attends to all data vectors and gives one output vector. Nothing in this computation depends on the data being produced by an RNN, or on the order of the data vectors. The next steps turn this idea into a layer of its own.

## Step 2: the attention layer, with one query (slides 30-35)

Using the names of the general operator:

| | Shape | |
|---|---|---|
| Query vector | $q$: $[D_Q]$ | was $s_{t-1}$ |
| Data vectors | $X$: $[N_X \times D_X]$ | were the $h_i$ |
| Similarities | $e$: $[N_X]$ | $e_i = f_{att}(q, X_i)$ |
| Attention weights | $a = \text{softmax}(e)$: $[N_X]$ | |
| Output vector | $y = \sum_i a_i X_i$: $[D_X]$ | was $c_t$ |

The first generalization (slide 36) replaces the learned $f_{att}$ with a plain dot product, which requires $D_Q = D_X$: $e_i = q \cdot X_i$. First, with an explicit loop:

In [ ]:
N_X, D_X = 4, 8
X = torch.randn(N_X, D_X)
q = torch.randn(D_X)

e = torch.zeros(N_X)
for i in range(N_X):
    e[i] = torch.dot(q, X[i])

a = torch.softmax(e, dim=0)

y = torch.zeros(D_X)
for i in range(N_X):
    y += a[i] * X[i]

print(f"e: {e}")
print(f"a: {a}")
print(f"y has shape {tuple(y.shape)}")

The same computation without loops. Each loop becomes one matrix product:

In [ ]:
e_matrix = X @ q  # [N_X x D_X] [D_X] => [N_X]
a_matrix = torch.softmax(e_matrix, dim=0)
y_matrix = a_matrix @ X  # [N_X] [N_X x D_X] => [D_X]

print(torch.allclose(y, y_matrix, atol=1e-6))

## Step 3: scaled dot product (slides 36-38)

**Change:** divide the similarities by $\sqrt{D_X}$, so that $e_i = q \cdot X_i / \sqrt{D_X}$.

Why? Slide 38 recalls that $a \cdot b = |a||b|\cos(\text{angle})$, and that the length of a vector grows with $\sqrt{D}$. So the dot product grows with the dimension. Large similarities make the softmax **saturate**: nearly all the weight goes to one input, and the gradients vanish.

Let's measure it. For each dimension, we compute the largest attention weight and the mean gradient that reaches the similarities, with and without the scale.

In [ ]:
def measure(D, scale, trials=200, N=16):
    largest, gradient = 0.0, 0.0
    for _ in range(trials):
        q = torch.randn(D)
        X = torch.randn(N, D)
        e = (X @ q) / (math.sqrt(D) if scale else 1.0)
        e.requires_grad_(True)
        a = torch.softmax(e, dim=0)
        a[0].backward()  # how much can the scores still change a weight?
        largest += a.max().item() / trials
        gradient += e.grad.abs().mean().item() / trials
    return largest, gradient


print(f"{'D':>6} | {'largest weight':>29} | {'mean gradient':>29}")
print(f"{'':>6} | {'no scale':>14} {'scaled':>14} | {'no scale':>14} {'scaled':>14}")
for D in [4, 64, 1024]:
    a_raw, g_raw = measure(D, scale=False)
    a_scaled, g_scaled = measure(D, scale=True)
    print(f"{D:>6} | {a_raw:>14.3f} {a_scaled:>14.3f} | {g_raw:>14.5f} {g_scaled:>14.5f}")

Without the scale, the largest weight approaches 1 as $D$ grows, and the gradient falls towards zero. With the scale, both stay about the same for any dimension.

## Step 4: multiple queries (slide 39)

**Change:** instead of one query, use a matrix of queries $Q$: $[N_Q \times D_X]$.

- Similarities: $E = QX^T / \sqrt{D_X}$, with shape $[N_Q \times N_X]$
- Attention weights: $A = \text{softmax}(E, \text{dim}=1)$, with shape $[N_Q \times N_X]$
- Output vectors: $Y = AX$, with shape $[N_Q \times D_X]$

`dim=1` means that the softmax runs over the data vectors: **each row of $A$ is a distribution, one for each query**. The slides draw the same matrix transposed, with the queries as columns, which is why they say that softmax normalizes each column.

In [ ]:
N_Q = 3
Q = torch.randn(N_Q, D_X)

E = Q @ X.T / math.sqrt(D_X)
A = torch.softmax(E, dim=1)
Y = A @ X

print(f"E: {tuple(E.shape)}, A: {tuple(A.shape)}, Y: {tuple(Y.shape)}")
print(f"each row of A sums to 1: {A.sum(dim=1)}")

# Each row of Y is what step 2 computes for one query
q0 = Q[0]
a0 = torch.softmax(X @ q0 / math.sqrt(D_X), dim=0)
print(torch.allclose(Y[0], a0 @ X, atol=1e-6))

## Step 5: keys and values (slides 40-45)

So far, $X$ plays two roles: it is compared with the queries, and it is what gets mixed in the output. These roles have different needs, so we separate them with two learned matrices.

**Change:** add a key matrix $W_K$: $[D_X \times D_Q]$ and a value matrix $W_V$: $[D_X \times D_V]$.

- Keys: $K = XW_K$, with shape $[N_X \times D_Q]$, are compared with the queries
- Values: $V = XW_V$, with shape $[N_X \times D_V]$, are what gets mixed
- Similarities: $E = QK^T / \sqrt{D_Q}$
- Attention weights: $A = \text{softmax}(E, \text{dim}=1)$
- Output vectors: $Y = AV$, with shape $[N_Q \times D_V]$

Now the dimensions can all be different:

In [ ]:
N_Q, D_Q = 3, 6
N_X, D_X = 4, 8
D_V = 5

Q = torch.randn(N_Q, D_Q)
X = torch.randn(N_X, D_X)
W_K = torch.randn(D_X, D_Q)
W_V = torch.randn(D_X, D_V)

K = X @ W_K
V = X @ W_V
E = Q @ K.T / math.sqrt(D_Q)
A = torch.softmax(E, dim=1)
Y = A @ V

for name, tensor in [("K", K), ("V", V), ("E", E), ("A", A), ("Y", Y)]:
    print(f"{name}: {tuple(tensor.shape)}")

## Step 6: cross-attention (slide 46)

The layer of step 5 has a name: **cross-attention**. There are two sets of vectors. Each query produces one output, which is a mix of information in the data vectors. In a translation model, the queries come from the sentence being generated and the data vectors from the sentence being read:

In [ ]:
def cross_attention(Q, X, W_K, W_V):
    K = X @ W_K
    V = X @ W_V
    E = Q @ K.T / math.sqrt(Q.shape[1])
    A = torch.softmax(E, dim=1)
    return A @ V, A


source = ["we", "see", "the", "sky"]
target = ["vediamo", "il", "cielo"]

Y, A = cross_attention(Q, X, W_K, W_V)
print(f"{len(target)} queries and {len(source)} data vectors give {Y.shape[0]} outputs")
show_matrix(A, target, source, "Cross-attention (random weights)", "data vectors", "queries")

## Step 7: self-attention (slides 47-51)

**Change:** there is no separate set of queries. Each input produces its own query, with a third matrix $W_Q$. From each input we compute a query, a key and a value.

- Queries: $Q = XW_Q$, with shape $[N \times D_{out}]$
- Keys: $K = XW_K$, with shape $[N \times D_{out}]$
- Values: $V = XW_V$, with shape $[N \times D_{out}]$
- Similarities: $E = QK^T / \sqrt{D_{out}}$, with shape $[N \times N]$
- Attention weights: $A = \text{softmax}(E, \text{dim}=1)$, with shape $[N \times N]$
- Output vectors: $Y = AV$, with shape $[N \times D_{out}]$

Each input produces one output, which is a mix of information from all inputs. The `mask` argument will only be used in step 10.

In [ ]:
def self_attention(X, W_Q, W_K, W_V, mask=None):
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    E = Q @ K.T / math.sqrt(Q.shape[1])
    if mask is not None:
        E = E.masked_fill(mask, float("-inf"))
    A = torch.softmax(E, dim=1)
    return A @ V, A


tokens = ["Attention", "is", "very", "cool"]
N, D_in, D_out = len(tokens), 8, 6

X = torch.randn(N, D_in)
W_Q = torch.randn(D_in, D_out) / math.sqrt(D_in)
W_K = torch.randn(D_in, D_out) / math.sqrt(D_in)
W_V = torch.randn(D_in, D_out) / math.sqrt(D_in)

Y, A = self_attention(X, W_Q, W_K, W_V)
print(f"X: {tuple(X.shape)}, A: {tuple(A.shape)}, Y: {tuple(Y.shape)}")
show_matrix(A, tokens, tokens, "Self-attention (random weights)", "keys", "queries")

## Step 8: permutation equivariance (slides 52-58)

Slides 52-57 permute the inputs and follow the computation: queries, keys, values, similarities, weights and outputs are all the same, only in a different order. Self-attention is **permutation equivariant**:

$$F(\sigma(X)) = \sigma(F(X))$$

This means that **self-attention works on sets of vectors**. Let's check it:

In [ ]:
permutation = torch.tensor([2, 0, 3, 1])

Y, _ = self_attention(X, W_Q, W_K, W_V)
Y_permuted, _ = self_attention(X[permutation], W_Q, W_K, W_V)

# F(sigma(X)) == sigma(F(X))
print(torch.allclose(Y_permuted, Y[permutation], atol=1e-5))

## Step 9: positional encoding (slides 59-60)

**Problem:** self-attention does not know the order of the sequence, so `very is cool Attention` would give the same outputs as `Attention is very cool`, only permuted.

**Solution:** add a positional encoding to each input, a vector that is a fixed function of the index. We use the sinusoids of the original Transformer: each dimension oscillates at a different frequency, so each position gets a different vector.

$$PE_{(pos, 2i)} = \sin(pos / 10000^{2i/D}) \qquad PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/D})$$

In [ ]:
def positional_encoding(N, D):
    position = torch.arange(N, dtype=torch.float32)[:, None]  # [N x 1]
    frequency = 10000 ** (-torch.arange(0, D, 2, dtype=torch.float32) / D)  # [D/2]
    encoding = torch.zeros(N, D)
    encoding[:, 0::2] = torch.sin(position * frequency)
    encoding[:, 1::2] = torch.cos(position * frequency)
    return encoding


plt.figure(figsize=(6, 3))
plt.imshow(positional_encoding(50, 64), aspect="auto", cmap="coolwarm")
plt.xlabel("dimension")
plt.ylabel("position")
plt.title("Each row is the vector of one position")
plt.colorbar()
plt.show()

The encoding depends only on the index, so it does not move when the words are permuted. Self-attention is no longer equivariant, and the outputs depend on the order:

In [ ]:
P = positional_encoding(N, D_in)

Y, _ = self_attention(X + P, W_Q, W_K, W_V)
Y_permuted, _ = self_attention(X[permutation] + P, W_Q, W_K, W_V)

print(torch.allclose(Y_permuted, Y[permutation], atol=1e-5))

Slide 60 shows a more recent alternative, RoPE, which rotates queries and keys according to their positions, so that the similarities depend on relative positions. We stay with the simpler option here.

## Step 10: masked self-attention (slides 61-62)

**Change:** don't let vectors "look ahead" in the sequence. We override the similarities with $-\infty$ wherever a query should not see a key. After the softmax, these positions get a weight of exactly 0. This controls which inputs each vector is allowed to look at.

It is used for language modeling, where you want to predict the next word: the output for `Attention` is trained to predict `is`, so it cannot see `is`.

In [ ]:
# True above the diagonal: key j comes after query i
mask = torch.triu(torch.ones(N, N, dtype=torch.bool), diagonal=1)
print(mask)

Y, A = self_attention(X + P, W_Q, W_K, W_V, mask=mask)
print(A)
show_matrix(A, tokens, tokens, "Masked self-attention", "keys", "queries")

Let's confirm that the future has no influence. Changing the last word must change only the last output:

In [ ]:
X_changed = X.clone()
X_changed[-1] = torch.randn(D_in)

Y_changed, _ = self_attention(X_changed + P, W_Q, W_K, W_V, mask=mask)

for i, token in enumerate(tokens):
    print(f"{token:>10}: output changed = {not torch.allclose(Y[i], Y_changed[i], atol=1e-6)}")

## Step 11: multi-headed self-attention (slides 63-67)

**Change:** run $H$ copies of self-attention in parallel, called **heads**, each with its own weights. Then stack up the $H$ independent outputs for each input, and mix them with an output matrix $W_O$: $[HD_H \times D]$.

Each head computes its own attention pattern, so one input can look at different inputs for different reasons. We start with an explicit loop, exactly as in the slides:

In [ ]:
H, D = 3, 12
D_H = D // H  # dimension of each head

X = torch.randn(N, D)

heads = []
for _ in range(H):
    heads.append({
        "W_Q": torch.randn(D, D_H) / math.sqrt(D),
        "W_K": torch.randn(D, D_H) / math.sqrt(D),
        "W_V": torch.randn(D, D_H) / math.sqrt(D),
    })
W_O = torch.randn(H * D_H, D) / math.sqrt(H * D_H)

outputs = []
attentions = []
for head in heads:
    Y_h, A_h = self_attention(X, head["W_Q"], head["W_K"], head["W_V"])
    outputs.append(Y_h)
    attentions.append(A_h)

Y = torch.cat(outputs, dim=1)  # stack the heads: [N x H*D_H]
O_loop = Y @ W_O  # mix the heads: [N x D]

print(f"each head: {tuple(outputs[0].shape)}, stacked: {tuple(Y.shape)}, output: {tuple(O_loop.shape)}")

fig, axes = plt.subplots(1, H, figsize=(9, 3))
for i, ax in enumerate(axes):
    ax.imshow(attentions[i], cmap="viridis", vmin=0, vmax=1)
    ax.set_xticks(range(N), tokens, rotation=45)
    ax.set_yticks(range(N), tokens)
    ax.set_title(f"Head {i + 1}")
plt.tight_layout()
plt.show()

## Step 12: self-attention is four matrix multiplies (slides 68-77)

In practice, all $H$ heads are computed in parallel with batched matrix multiplies. The whole layer comes down to four products:

| | Operation | Shapes |
|---|---|---|
| 1 | QKV projection | $[N \times D]\,[D \times 3HD_H] \Rightarrow [N \times 3HD_H]$ |
| | split and reshape | $Q, K, V$: $[H \times N \times D_H]$ |
| 2 | QK similarity | $[H \times N \times D_H]\,[H \times D_H \times N] \Rightarrow [H \times N \times N]$ |
| 3 | V-weighting | $[H \times N \times N]\,[H \times N \times D_H] \Rightarrow [H \times N \times D_H]$ |
| | reshape | $[N \times HD_H]$ |
| 4 | Output projection | $[N \times HD_H]\,[HD_H \times D] \Rightarrow [N \times D]$ |

We put the weights of all heads in one matrix, and follow the table line by line. The result must match the loop above.

In [ ]:
# One matrix with the weights of all heads: [D x 3*H*D_H]
W_Q_all = torch.cat([head["W_Q"] for head in heads], dim=1)
W_K_all = torch.cat([head["W_K"] for head in heads], dim=1)
W_V_all = torch.cat([head["W_V"] for head in heads], dim=1)
W_QKV = torch.cat([W_Q_all, W_K_all, W_V_all], dim=1)

# 1. QKV projection
QKV = X @ W_QKV
print(f"1. QKV: {tuple(QKV.shape)}")

# Split and reshape: [N x H*D_H] => [N x H x D_H] => [H x N x D_H]
Q, K, V = QKV.chunk(3, dim=1)
Q = Q.reshape(N, H, D_H).transpose(0, 1)
K = K.reshape(N, H, D_H).transpose(0, 1)
V = V.reshape(N, H, D_H).transpose(0, 1)
print(f"   Q, K, V: {tuple(Q.shape)}")

# 2. QK similarity
E = Q @ K.transpose(1, 2) / math.sqrt(D_H)
A = torch.softmax(E, dim=2)
print(f"2. E, A: {tuple(E.shape)}")

# 3. V-weighting
Y = A @ V
print(f"3. Y: {tuple(Y.shape)}")

# Reshape: [H x N x D_H] => [N x H x D_H] => [N x H*D_H]
Y = Y.transpose(0, 1).reshape(N, H * D_H)
print(f"   Y: {tuple(Y.shape)}")

# 4. Output projection
O = Y @ W_O
print(f"4. O: {tuple(O.shape)}")

print(f"same result as the loop: {torch.allclose(O, O_loop, atol=1e-5)}")

## Step 13: memory cost (slides 78-79)

Slide 76 asks how much memory this takes as $N$ grows. The answer is in the attention weights: $A$ has shape $[H \times N \times N]$, so it grows with $O(N^2)$. For $N$ = 100,000 and $H$ = 64, with 2 bytes per number:

In [ ]:
H_big = 64
for N_big in [1_000, 10_000, 100_000]:
    size = H_big * N_big * N_big * 2  # bytes
    print(f"N = {N_big:>7,}: {size / 2**30:>10,.2f} GiB")

That's the 1.192 TB of slide 78: GPUs don't have that much memory. The **Flash Attention** algorithm (slide 79) computes steps 2 and 3 at the same time, without storing the full attention matrix. PyTorch provides it in `F.scaled_dot_product_attention`, which gives the same result as our steps 2 and 3:

In [ ]:
Y_flash = F.scaled_dot_product_attention(Q, K, V)
Y_flash = Y_flash.transpose(0, 1).reshape(N, H * D_H)

print(torch.allclose(Y_flash, Y, atol=1e-5))

## Step 14: a PyTorch module

To train the weights, we turn step 12 into a module. Two changes:

- The weights become `nn.Linear` layers, so that PyTorch tracks them.
- A batch dimension $B$ goes in front of everything, so the shapes become $[B \times H \times N \times D_H]$.

This is the class that the `Transformer from Scratch` notebook uses.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, D, H):
        super().__init__()
        assert D % H == 0, "D must be divisible by H"
        self.H = H
        self.D_H = D // H
        self.qkv = nn.Linear(D, 3 * D)  # 1. QKV projection
        self.out = nn.Linear(D, D)  # 4. output projection

    def forward(self, X, mask=None):
        B, N, D = X.shape

        # 1. QKV projection, split and reshape
        Q, K, V = self.qkv(X).chunk(3, dim=2)  # each [B x N x D]
        Q = Q.reshape(B, N, self.H, self.D_H).transpose(1, 2)  # [B x H x N x D_H]
        K = K.reshape(B, N, self.H, self.D_H).transpose(1, 2)
        V = V.reshape(B, N, self.H, self.D_H).transpose(1, 2)

        # 2. QK similarity
        E = Q @ K.transpose(2, 3) / math.sqrt(self.D_H)  # [B x H x N x N]
        if mask is not None:
            E = E.masked_fill(mask, float("-inf"))
        A = torch.softmax(E, dim=3)

        # 3. V-weighting, and reshape
        Y = A @ V  # [B x H x N x D_H]
        Y = Y.transpose(1, 2).reshape(B, N, D)

        # 4. Output projection
        return self.out(Y), A

PyTorch has a ready-made version of this layer, `nn.MultiheadAttention`. If we copy our weights into it, the outputs must be the same:

In [ ]:
B, N, D, H = 2, 5, 16, 4
X = torch.randn(B, N, D)

ours = MultiHeadSelfAttention(D, H)
pytorch = nn.MultiheadAttention(D, H, batch_first=True)

with torch.no_grad():
    pytorch.in_proj_weight.copy_(ours.qkv.weight)
    pytorch.in_proj_bias.copy_(ours.qkv.bias)
    pytorch.out_proj.weight.copy_(ours.out.weight)
    pytorch.out_proj.bias.copy_(ours.out.bias)

Y_ours, _ = ours(X)
Y_pytorch, _ = pytorch(X, X, X)

print(f"output: {tuple(Y_ours.shape)}")
print(f"same result as nn.MultiheadAttention: {torch.allclose(Y_ours, Y_pytorch, atol=1e-5)}")

## Summary

Each step added one change to the previous one:

| Layer | Queries | Compared with | Mixed in the output |
|---|---|---|---|
| Attention in an RNN | decoder state | encoder states | encoder states |
| Attention layer | given | data vectors $X$ | data vectors $X$ |
| With keys and values (cross-attention) | given | keys $XW_K$ | values $XW_V$ |
| Self-attention | $XW_Q$ | keys $XW_K$ | values $XW_V$ |

Slide 84 compares the three ways of processing sequences:

- **RNN:** works on ordered sequences, but is hard to parallelize.
- **Convolution:** works on grids, but needs many layers to relate distant positions.
- **Self-attention:** works on sets of vectors, each output depends directly on all inputs, and it is highly parallel, at the cost of memory that grows with $N^2$.

### Try it yourself

1. In step 3, add a line for `D = 16384`. What happens to the gradient without the scale?
2. In step 10, change the mask so that each word sees only itself and the previous word. What does the attention matrix look like?
3. In step 8, repeat the check with the mask of step 10. Is masked self-attention still permutation equivariant? Why?

### References

- [CS231n, Lecture 8: Attention and Transformers](https://cs231n.stanford.edu/slides/2026/lecture_8.pdf)
- S. Raschka, *Build a Large Language Model (From Scratch)*, chapter 3: the step-by-step style of this notebook was inspired by it.